<a href="https://colab.research.google.com/github/asdp132A3a/alt-tab-macos/blob/master/fsrs4anki_optimizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FSRS4Anki v6.1.3 Optimizer

[![open in colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/open-spaced-repetition/fsrs4anki/blob/v6.1.3/fsrs4anki_optimizer.ipynb)

↑ Click the above button to open the optimizer on Google Colab.

> If you can't see the button and are located in the Chinese Mainland, please use a proxy or VPN.

Upload your **Anki Deck Package (.apkg)** file or **Anki Collection Package (.colpkg)** file on the `Left sidebar -> Files`, drag and drop your file in the current directory (not the `sample_data` directory).

> Add blockquote



No need to include media. Need to include scheduling information.

> If you use the latest version of Anki, please check the box `Support older Anki versions (slower/larger files)` when you export.

You can export it via `File -> Export...` or `Ctrl + E` in the main window of Anki.

Then replace the `filename` with yours in the next code cell. And set the `timezone` and `next_day_starts_at` which can be found in your preferences of Anki.

After that, just run all (`Runtime -> Run all` or `Ctrl + F9`) and wait for minutes. You can see the optimal parameters in section **2.3 Result**. Copy them, replace the parameters in `fsrs4anki_scheduler.js`, and paste them into the custom scheduling of your deck options (require Anki version >= 2.1.55).

**NOTE**: The default output is generated from my review logs. If you find the output is the same as mine, maybe your notebook hasn't run there.

**Contribute to SRS Research**: If you want to share your data with me, please fill this form: https://forms.gle/KaojsBbhMCytaA7h8

In [ ]:
# Here are some settings that you need to replace before running this optimizer.

filename = "collection-2026-05-11@16-24-30.colpkg"
# If you upload deck file, replace it with your deck filename. E.g., ALL__Learning.apkg
# If you upload collection file, replace it with your colpkg filename. E.g., collection-2022-09-18@13-21-58.colpkg

# Replace it with your timezone. I'm in China, so I use Asia/Shanghai.
# You can find your timezone here: https://gist.github.com/heyalexej/8bf688fd67d7199be4a1682b3eec7568
timezone = 'US/Central'

# Replace it with your Anki's setting in Preferences -> Scheduling.
next_day_starts_at = 2

# Replace it if you don't want the optimizer to use the review logs before a specific date.
revlog_start_date = "2006-10-05"  # YYYY-MM-DD

# Set it to True if you don't want the optimizer to use the review logs from suspended cards.
filter_out_suspended_cards = False

# Red: 1, Orange: 2, Green: 3, Blue: 4, Pink: 5, Turquoise: 6, Purple: 7
# Set it to [1, 2] if you don't want the optimizer to use the review logs from cards with red or orange flag.
filter_out_flags = []

enable_short_term = True

recency_weight = True

## 1 Build dataset

### 1.1 Extract Anki collection & deck file

In [ ]:
%pip install -q fsrs_optimizer==6.1.5
# for local development
# import os
# import sys
# sys.path.insert(0, os.path.abspath('../fsrs-optimizer/src/fsrs_optimizer/'))
import fsrs_optimizer as optimizer
optimizer = optimizer.Optimizer(enable_short_term=enable_short_term)
optimizer.anki_extract(filename, filter_out_suspended_cards, filter_out_flags)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 926.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 17.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
access 1.1.10.post3 requires scipy>=1.14.1, but you have scipy 1.14.0 which is incompatible.
Deck file extracted successfully!
revlog.csv saved.


### 1.2 Create time-series feature & analysis

The following code cell will extract the review logs from your Anki collection and preprocess them to a trainset which is saved in [./revlog_history.tsv](./revlog_history.tsv).

The time-series features are important in optimizing the model's parameters. For more detail, please see my paper: https://www.maimemo.com/paper/

Then it will generate a concise analysis for your review logs.

- The `r_history` is the history of ratings on each review. `3,3,3,1` means that you press `Good, Good, Good, Again`. It only contains the first rating for each card on the review date, i.e., when you press `Again` in review and  `Good` in relearning steps 10min later, only `Again` will be recorded.
- The `avg_interval` is the actual average interval after you rate your cards as the `r_history`. It could be longer than the interval given by Anki's built-in scheduler because you reviewed some overdue cards.
- The `avg_retention` is the average retention after you press as the `r_history`. `Again` counts as failed recall, and `Hard, Good and Easy` count as successful recall. Retention is the percentage of your successful recall.
- The `stability` is the estimated memory state variable, which is an approximate interval that leads to 90% retention.
- The `factor` is `stability / previous stability`.
- The `group_cnt` is the number of review logs that have the same `r_history`.

In [ ]:
analysis = optimizer.create_time_series(
    timezone, revlog_start_date, next_day_starts_at)
print(analysis)

  0%|          | 0/2474 [00:00<?, ?it/s]

Trainset saved.
Retention calculated.


  0%|          | 0/4623 [00:00<?, ?it/s]

Stability calculated.


analysis:   0%|          | 0/132 [00:00<?, ?it/s]

Analysis saved!
1:again, 2:hard, 3:good, 4:easy
first_rating  i       r_history  avg_interval  avg_retention  stability  factor  group_cnt
           1  2             (1)           1.1          0.252        0.0     NaN        995
           1  3           (1),3           1.2          0.854        0.7     inf        213
           1  4         (1),3,3           3.6          0.953       11.2   16.00        176
           1  5       (1),3,3,3          10.5          0.922       15.1    1.35        165
           1  6     (1),3,3,3,3          27.2          0.970       75.4    4.99        147
           1  7   (1),3,3,3,3,3          56.0          0.943      126.9    1.68        117
           3  2             (3)           1.2          0.887        1.2     inf        701
           3  2           (3,3)           7.8          0.911        6.7     inf        114
           3  3           (3),3           5.6          0.965       16.4   13.67        589
           3  3         (3,3),3          1

## 2 Optimize parameter

### 2.1 Define & Train the model

FSRS is a time-series model for predicting memory states.

The [./revlog_history.tsv](./revlog_history.tsv) generated before will be used for training the FSRS model.

In [ ]:
optimizer.define_model()
optimizer.pretrain(verbose=False)
optimizer.train(verbose=False, recency_weight=recency_weight)

  0%|          | 0/14996 [00:00<?, ?it/s]

[]

### 2.2 Result

Copy the optimal parameters for FSRS for you in the output of next code cell after running.

In [ ]:
print(optimizer.w)

[0.026, 0.0733, 1.6989, 10.7359, 6.4378, 0.7959, 3.1697, 0.1638, 1.9106, 0.2938, 0.8081, 1.2366, 0.1117, 0.1888, 1.2778, 0.6077, 1.8445, 1.2091, 0.486, 0.3643, 0.4249]


<font color=orange>Note: These values should be used with build-in FSRS of Anki 23.12 or custom scheduling script of FSRS4Anki v4.11.0</font>

### 2.3 Preview

You can see the memory states and intervals generated by FSRS as if you press the good in each review at the due date scheduled by FSRS.

In [ ]:
requestRetention = 0.9  # recommended setting: 0.8 ~ 0.9

preview = optimizer.preview(requestRetention)
print(preview)

1:again, 2:hard, 3:good, 4:easy

first rating: 1
rating history: (1,3,3),3,3,3,3,3,3,3,3
interval history: 0.0d,0.0d,0.0d,1.0d,5.0d,20.0d,2.2m,5.8m,1.1y,2.1y,3.9y
factor history: 0.0,0.0,0.0,0.0,5.00,4.00,3.30,2.65,2.25,1.98,1.80
difficulty history: 0,6.4,4.8,3.5,2.3,1.4,1.0,1.0,1.0,1.0,1.0
stability history: 0,0.0,0.2,0.6,5.2,20.5,65.6,175.4,393.5,780.7,1408.3

first rating: 2
rating history: (2,3,3),3,3,3,3,3,3,3,3
interval history: 0.0d,0.0d,0.0d,1.0d,6.0d,25.0d,2.7m,6.9m,1.2y,2.4y,4.3y
factor history: 0.0,0.0,0.0,0.0,6.00,4.17,3.20,2.58,2.19,1.94,1.78
difficulty history: 0,5.2,3.8,2.6,1.6,1.0,1.0,1.0,1.0,1.0,1.0
stability history: 0,0.1,0.3,0.9,5.8,24.8,80.1,205.6,450.6,876.5,1557.7

first rating: 3
rating history: (3,3),3,3,3,3,3,3,3,3,3
interval history: 0.0d,0.0d,3.0d,15.0d,1.8m,4.9m,11.3m,1.9y,3.5y,5.9y,9.4y
factor history: 0.0,0.0,0.0,5.00,3.53,2.77,2.31,2.03,1.83,1.70,1.60
difficulty history: 0,2.5,1.5,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
stability history: 0,1.7,2.5,14.5,53.2,147

You can change the `test_rating_sequence` to see the scheduling intervals in different ratings.

In [ ]:
test_rating_sequence = "3,3,3,3,3,1,1,3,3,3,3,3"
requestRetention = 0.9  # recommended setting: 0.8 ~ 0.9

preview_sequence = optimizer.preview_sequence(
    test_rating_sequence, requestRetention)
print(preview_sequence)

rating history: 3,3,3,3,3,1,1,3,3,3,3,3
interval history: 0.0d,2.0d,10.0d,1.2m,3.7m,8.9m,3.0d,1.0d,3.0d,10.0d,1.1m,2.9m,7.2m
factor history: 0.0,0.0,5.00,3.70,2.97,2.43,0.01,0.33,3.00,3.33,3.20,2.75,2.44
difficulty history: 0,2.5,1.5,1.0,1.0,1.0,5.6,6.7,5.0,3.6,2.5,1.5,1.0


### 2.4 Predict memory states and distribution of difficulty

Predict memory states for each review group and save them in [./prediction.tsv](./prediction.tsv).

Meanwhile, it will count the distribution of difficulty.

In [ ]:
optimizer.predict_memory_states()

,count
difficulty,
1,0.344158
2,0.127034
3,0.124633
4,0.066885
5,0.063750
6,0.092358
7,0.181182
